In [ ]:
# White Christmas probability map and predictive snow cover model
# Part 1: Historical analysis (1980-2024) using ERA5-Land
# Part 2: Statistical snow model training
# Part 3: Future projections using CMIP6 SSP2-4.5 scenarios

import numpy as np
import xarray as xr
import zipfile
import tempfile
import os
from pathlib import Path
from typing import List, Optional

# Restore deprecated numpy aliases for rioxarray compatibility
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool"):
    np.bool = bool

import rioxarray

## 1. Multi-Year Data Loader

Load all ERA5-Land annual zip files (1980-2024) and concatenate into a unified dataset. Includes temperature (t2m), precipitation (tp), and snow depth (sde).

In [ ]:
def load_era5_multiyear(
    base_path: str,
    years: List[int],
    variables: Optional[List[str]] = None,
) -> xr.Dataset:
    """
    Load and concatenate ERA5-Land annual zip files into a unified dataset.
    
    Args:
        base_path: Directory containing zip files
        years: List of years to load (e.g., range(1980, 2025))
        variables: Optional list of variables to keep (e.g., ['sde', 't2m', 'tp'])
    
    Returns:
        xr.Dataset with concatenated time dimension (lazy-loaded with dask)
    """
    datasets = []
    temp_files = []  # Track extracted files to clean up later
    
    for year in years:
        zip_path = Path(base_path) / f"era5land_white_christmas_dec10-26_{year}.zip"
        
        if not zip_path.exists():
            print(f"Warning: {zip_path.name} not found, skipping...")
            continue
        
        print(f"Loading {year}...", end=" ")
        
        # Extract NetCDF to a persistent temporary directory
        # Note: We'll keep temp files until concatenation is complete
        with zipfile.ZipFile(zip_path) as zf:
            # Find the NetCDF file inside
            nc_files = [n for n in zf.namelist() if n.endswith('.nc')]
            if not nc_files:
                print(f"No .nc file found in {zip_path.name}")
                continue
            
            nc_name = nc_files[0]
            
            # Create a persistent temp directory for this year
            tmpdir = tempfile.mkdtemp()
            zf.extract(nc_name, path=tmpdir)
            extracted_path = os.path.join(tmpdir, nc_name)
            temp_files.append((tmpdir, extracted_path))
            
            # Open dataset with chunking (KEEP IT LAZY - don't call .load())
            ds_year = xr.open_dataset(
                extracted_path, 
                engine='netcdf4',
                chunks={'time': 50}  # Chunk time dimension for efficient processing
            )
            
            # Optionally filter variables
            if variables:
                ds_year = ds_year[variables]
            
            datasets.append(ds_year)
            print("✓")
    
    if not datasets:
        raise ValueError("No datasets were loaded successfully")
    
    # Concatenate along time dimension (remains lazy with dask)
    print(f"\nConcatenating {len(datasets)} years (lazy)...")
    ds_merged = xr.concat(datasets, dim='valid_time')
    
    # Rename time dimension for consistency
    if 'valid_time' in ds_merged.dims:
        ds_merged = ds_merged.rename({'valid_time': 'time'})
    
    print(f"Final shape: {dict(ds_merged.dims)}")
    print("Note: Dataset is lazy-loaded. Use .compute() or .load() only when needed.")
    
    # Clean up temp files after a delay (or rely on OS cleanup)
    # For now, we'll leave them - they'll be auto-cleaned on exit
    # Future improvement: use atexit to register cleanup
    
    return ds_merged

In [ ]:
# Load all ERA5-Land data (1980-2024)
era5_base_path = r"C:\Users\AlexanderKoch\TransparentEarth\lx3\data\ERA5land"
years_to_load = list(range(1980, 2025))

ds_era5 = load_era5_multiyear(
    base_path=era5_base_path,
    years=years_to_load,
    variables=['sde', 't2m', 'tp']  # snow depth, 2m temperature, total precipitation
)

In [ ]:
# Inspect the merged dataset
ds_era5

## 2. Load Static Fields

Load geopotential height (elevation), high vegetation cover, and low vegetation cover. These will be used for snow model features.

In [ ]:
# Load static fields
geo_path = r"C:\Users\AlexanderKoch\TransparentEarth\lx3\data\ERA5land\geo.area-subset.90.180.0.-180.nc"
cvh_path = r"C:\Users\AlexanderKoch\TransparentEarth\lx3\data\ERA5land\cvh.area-subset.90.180.0.-180.nc"
cvl_path = r"C:\Users\AlexanderKoch\TransparentEarth\lx3\data\ERA5land\cvl.area-subset.90.180.0.-180.nc"

# Load geopotential (convert to elevation in meters: geo / 9.80665)
ds_geo = xr.open_dataset(geo_path)
elevation = ds_geo['z'] / 9.80665  # geopotential to meters
elevation.name = 'elevation'

# Load vegetation cover fractions
ds_cvh = xr.open_dataset(cvh_path)
cvh = ds_cvh['cvh']  # high vegetation cover (0-1)

ds_cvl = xr.open_dataset(cvl_path)
cvl = ds_cvl['cvl']  # low vegetation cover (0-1)

print(f"Elevation shape: {elevation.shape}")
print(f"CVH shape: {cvh.shape}")
print(f"CVL shape: {cvl.shape}")

# Convert longitude from 0-360 to -180-180 and remove duplicate
def convert_lon_0360_to_pm180(da):
    """
    Convert DataArray longitude from 0-360 to -180-180 range.
    Removes duplicate at wraparound (0° = 360°).
    Handles duplicate 180° values that result from conversion.
    """
    if 'longitude' in da.dims:
        # Check if longitude is in 0-360 range
        lon = da['longitude'].values
        if lon.min() >= 0 and lon.max() > 180:
            print(f"  Converting longitude from 0-360 to -180-180...")
            
            # Remove duplicate at 360° (same as 0°)
            if len(lon) > 1 and (lon[-1] - lon[0]) > 359:
                da = da.isel(longitude=slice(0, -1))
                lon = da['longitude'].values
            
            # Convert: 0-180 stays same, 180-360 becomes -180-0
            lon_converted = np.where(lon > 180, lon - 360, lon)
            da = da.assign_coords(longitude=lon_converted)
            
            # Remove duplicate 180° values (keep first, drop second)
            _, unique_idx = np.unique(da['longitude'].values, return_index=True)
            unique_idx = np.sort(unique_idx)  # Maintain original order
            
            if len(unique_idx) < len(da['longitude']):
                print(f"  Removing {len(da['longitude']) - len(unique_idx)} duplicate longitude value(s)...")
                da = da.isel(longitude=unique_idx)
            
            # Sort by longitude for proper ordering
            da = da.sortby('longitude')
            print(f"  ✓ Longitude converted. New range: {da['longitude'].values.min():.1f} to {da['longitude'].values.max():.1f}")
    
    return da

# Apply conversion
elevation = convert_lon_0360_to_pm180(elevation)
cvh = convert_lon_0360_to_pm180(cvh)
cvl = convert_lon_0360_to_pm180(cvl)

print(f"\nAfter conversion:")
print(f"Elevation shape: {elevation.shape}")
print(f"CVH shape: {cvh.shape}")
print(f"CVL shape: {cvl.shape}")

In [ ]:
cvl.plot()

In [ ]:
# Align static fields to ERA5 grid if needed
# Broadcast to match ERA5 spatial dimensions for later use
def align_to_era5(static_field, template_ds):
    """
    Align static field to ERA5 spatial grid using reindexing.
    Falls back to nearest-neighbor if grids don't match exactly.
    """
    # Check if coordinates match ERA5
    if 'latitude' in static_field.dims and 'longitude' in static_field.dims:
        try:
            # Try exact reindex first (fastest if grids match)
            aligned = static_field.reindex(
                latitude=template_ds['latitude'],
                longitude=template_ds['longitude'],
                method='nearest',
                tolerance=0.1  # Allow small tolerance (0.1 degree)
            )
            return aligned
        except Exception as e:
            print(f"Reindex failed: {e}, attempting alternative alignment...")
            
            # Fallback: Use sel with nearest neighbor
            try:
                aligned = static_field.sel(
                    latitude=template_ds['latitude'],
                    longitude=template_ds['longitude'],
                    method='nearest'
                )
                return aligned
            except Exception as e2:
                print(f"Alignment failed: {e2}. Returning original field.")
                return static_field
    else:
        return static_field

# Align static fields to ERA5 grid
elevation_aligned = align_to_era5(elevation, ds_era5)
cvh_aligned = align_to_era5(cvh, ds_era5)
cvl_aligned = align_to_era5(cvl, ds_era5)

print("Static fields aligned to ERA5 grid")

## 3. Historical White Christmas Probability Analysis

Compute white Christmas probability and trends for multiple 10-year intervals:
- 1985-1994
- 1995-2004
- 2005-2014
- 2015-2024

In [ ]:
# Configuration
var = "sde"                 # snow depth variable name
threshold_m = 0.01         # 1 cm in meters

In [ ]:
def white_christmas_prob_and_trend(
    ds: xr.Dataset,
    var: str = "sde",
    time_dim: str = "time",
    threshold_m: float = 0.01,
    start_year: int = 1980,
    end_year: int = 2024,
    require_all_3_days: bool = True,
    min_years_for_trend: int = 8,
) -> xr.Dataset:
    """
    Compute white-Christmas probability and trend for a given year range.

    Definition (per grid cell, per year):
      white_christmas_year = max snow depth over Dec 24–26 >= threshold_m

    Outputs:
      - p_white_pct: probability in percent (0..100)
      - n_years: number of valid years used
      - trend_pp_per_decade: OLS trend in probability (percentage points per decade)

    Assumes regular lat/lon grid with dims latitude/longitude and a datetime time dimension.
    """
    da = ds[var]

    # --- Filter to target years and Dec 24–26 ---
    da = da.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))
    da = da.where(
        (da["time"].dt.month == 12) & (da["time"].dt.day.isin([24, 25, 26])),
        drop=True
    )

    # --- Per-year max snow depth across the 3 days ---
    sd_year_max = da.groupby("time.year").max("time", skipna=True)  # dims: year, lat, lon

    # --- Optional: require all 3 days present per year and cell ---
    if require_all_3_days:
        days_present = da.notnull().groupby("time.year").sum("time")  # 0..3
        sd_year_max = sd_year_max.where(days_present >= 3)

    # --- Binary white Christmas per year (0/1 float for math) ---
    white = (sd_year_max >= threshold_m).astype("float32")

    # --- Probability and sample size ---
    p_white_pct = 100 * white.mean("year", skipna=True)
    n_years = white.count("year")

    # =========================
    # Trend per grid cell (OLS slope)
    # Units: percentage points per decade
    # =========================
    years = white["year"].values.astype(np.float64)
    x = years - years.mean()
    x_da = xr.DataArray(x, dims=["year"], coords={"year": white["year"]})

    # NaN-robust sums
    valid = white.notnull()
    n = valid.sum("year")

    x_masked = x_da.where(valid)
    Sx  = x_masked.sum("year")
    Sy  = white.sum("year", skipna=True)
    Sxx = (x_masked * x_masked).sum("year")
    Sxy = (x_masked * white).sum("year", skipna=True)

    den = (Sxx - (Sx * Sx) / n)
    num = (Sxy - (Sx * Sy) / n)

    slope_per_year = num / den
    trend_pp_per_decade = slope_per_year * 100 * 10

    # Mask noisy/degenerate cells
    trend_pp_per_decade = trend_pp_per_decade.where(
        (n >= min_years_for_trend) & np.isfinite(den) & (den != 0)
    )

    return xr.Dataset(
        {
            "p_white_pct": p_white_pct,
            "n_years": n_years,
            "trend_pp_per_decade": trend_pp_per_decade,
        }
    )

In [ ]:
# Compute white Christmas statistics for multiple periods
periods = [
    (1985, 1994, "1985_1994"),
    (1995, 2004, "1995_2004"),
    (2005, 2014, "2005_2014"),
    (2015, 2024, "2015_2024"),
]

results = {}
for start, end, label in periods:
    print(f"Computing {label}...")
    results[label] = white_christmas_prob_and_trend(
        ds_era5, 
        start_year=start, 
        end_year=end, 
        min_years_for_trend=8
    )

# Also compute full period (1980-2024) for baseline
print("Computing full period 1980-2024...")
results["1980_2024"] = white_christmas_prob_and_trend(
    ds_era5,
    start_year=1980,
    end_year=2024,
    min_years_for_trend=8
)

In [ ]:
def to_geotiff_ready(da):
    da = da.rename({"longitude": "x", "latitude": "y"})
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    da = da.rio.write_crs("EPSG:4326", inplace=False)

    # Ensure north-up (y decreasing)
    if da.y[0] < da.y[-1]:
        da = da.sortby("y", ascending=False)

    return da

# Export all probability and trend maps as GeoTIFFs
for period_label, result_ds in results.items():
    print(f"\nExporting {period_label}...")
    
    # Probability map
    p_white_tif = to_geotiff_ready(result_ds["p_white_pct"])
    p_white_tif.rio.to_raster(
        f"white_christmas_probability_pct_{period_label}.tif",
        compress="DEFLATE",
        dtype="float32",
        nodata=np.nan,
    )
    
    # Trend map
    trend_tif = to_geotiff_ready(result_ds["trend_pp_per_decade"])
    trend_tif.rio.to_raster(
        f"white_christmas_trend_pp_per_decade_{period_label}.tif",
        compress="DEFLATE",
        dtype="float32",
        nodata=np.nan,
    )
    
    # Valid years count
    n_years_tif = to_geotiff_ready(result_ds["n_years"])
    n_years_tif.rio.to_raster(
        f"white_christmas_valid_years_{period_label}.tif",
        compress="DEFLATE",
        dtype="float32",
        nodata=np.nan,
    )

print("\n✓ All GeoTIFFs exported successfully")

## 4. Feature Engineering for Snow Cover Model

Create training features from ERA5-Land data:
- Temperature statistics (mean, min)
- Accumulated precipitation (where T < 2°C = snow)
- Positive degree-days (PDD)
- Elevation-corrected temperature
- Vegetation cover fractions

In [ ]:
def engineer_snow_features(
    ds: xr.Dataset,
    elevation: xr.DataArray,
    cvh: xr.DataArray,
    cvl: xr.DataArray,
    pre_christmas_days: int = 14,  # Dec 10-23
) -> xr.Dataset:
    """
    Create predictive features for snow cover modeling.
    
    Features:
    - t2m_mean_dec24_26: Mean temperature Dec 24-26
    - t2m_min_dec24_26: Minimum temperature Dec 24-26
    - precip_snow_accumulated: Accumulated precipitation where T<2°C (Dec 10-23)
    - pdd_pre_christmas: Positive degree-days before Christmas (Dec 10-23)
    - t2m_elevation_corrected: Temperature corrected for elevation
    - elevation: Elevation in meters
    - cvh: High vegetation cover fraction
    - cvl: Low vegetation cover fraction
    - total_vegetation: Total vegetation cover
    
    Target:
    - snow_presence: Binary indicator (sde >= 0.01m) on Dec 24-26
    """
    
    # Extract variables
    t2m = ds['t2m'] - 273.15  # Convert Kelvin to Celsius
    tp = ds['tp'] * 1000  # Convert m to mm
    sde = ds['sde']
    
    # Filter to December only
    dec_data = t2m.where(t2m['time'].dt.month == 12, drop=True)
    tp_dec = tp.where(tp['time'].dt.month == 12, drop=True)
    sde_dec = sde.where(sde['time'].dt.month == 12, drop=True)
    
    # Create year-based features
    years = np.unique(dec_data['time'].dt.year.values)
    
    feature_list = []
    
    for year in years:
        print(f"Engineering features for {year}...", end=" ")
        
        # Get data for this year
        t2m_year = dec_data.sel(time=f"{year}")
        tp_year = tp_dec.sel(time=f"{year}")
        sde_year = sde_dec.sel(time=f"{year}")
        
        # --- Dec 24-26 temperature features ---
        t2m_dec24_26 = t2m_year.where(
            t2m_year['time'].dt.day.isin([24, 25, 26]), 
            drop=True
        )
        
        if len(t2m_dec24_26['time']) == 0:
            print("No Dec 24-26 data, skipping")
            continue
            
        t2m_mean = t2m_dec24_26.mean('time', skipna=True)
        t2m_min = t2m_dec24_26.min('time', skipna=True)
        
        # --- Pre-Christmas precipitation (Dec 10-23) ---
        t2m_pre = t2m_year.where(
            (t2m_year['time'].dt.day >= 10) & (t2m_year['time'].dt.day <= 23),
            drop=True
        )
        tp_pre = tp_year.where(
            (tp_year['time'].dt.day >= 10) & (tp_year['time'].dt.day <= 23),
            drop=True
        )
        
        # Classify precipitation as snow where T < 2°C
        is_snow = t2m_pre < 2.0
        precip_snow = (tp_pre * is_snow).sum('time', skipna=True)
        
        # --- Positive degree-days (pre-Christmas) ---
        pdd = (t2m_pre.clip(min=0)).sum('time', skipna=True)
        
        # --- Elevation-corrected temperature (lapse rate: 6.5°C/1000m) ---
        t2m_corrected = t2m_mean + (elevation_aligned * 0.0065)
        
        # --- Target: Snow presence on Dec 24-26 ---
        sde_dec24_26 = sde_year.where(
            sde_year['time'].dt.day.isin([24, 25, 26]),
            drop=True
        )
        snow_presence = (sde_dec24_26.max('time', skipna=True) >= 0.01).astype('float32')
        
        # Combine features for this year
        year_features = xr.Dataset({
            't2m_mean_dec24_26': t2m_mean,
            't2m_min_dec24_26': t2m_min,
            'precip_snow_accumulated': precip_snow,
            'pdd_pre_christmas': pdd,
            't2m_elevation_corrected': t2m_corrected,
            'elevation': elevation_aligned,
            'cvh': cvh_aligned,
            'cvl': cvl_aligned,
            'total_vegetation': cvh_aligned + cvl_aligned,
            'snow_presence': snow_presence,
        })
        
        # Add year coordinate
        year_features = year_features.assign_coords({'year': year})
        year_features = year_features.expand_dims('year')
        
        feature_list.append(year_features)
        print("✓")
    
    # Concatenate all years
    print(f"\nConcatenating {len(feature_list)} years...")
    features_ds = xr.concat(feature_list, dim='year')
    
    return features_ds

In [ ]:
# Generate features for all years (1980-2024)
features_ds = engineer_snow_features(
    ds_era5,
    elevation_aligned,
    cvh_aligned,
    cvl_aligned
)

In [ ]:
# Inspect features
features_ds

## 5. Model Training and Validation

Train three snow cover models on 1980-2014 data, validate on 2015-2024 holdout:
- **Logistic Regression**: Fast, interpretable
- **Random Forest**: Non-linear, feature importance
- **Physical Degree-Day Model**: Generalizes to unseen climates

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, f1_score, brier_score_loss
import matplotlib.pyplot as plt

# Prepare training and validation data
# Training: 1980-2014
# Validation: 2015-2024

train_years = features_ds.where(features_ds['year'] <= 2014, drop=True)
val_years = features_ds.where(features_ds['year'] >= 2015, drop=True)

# Feature list (exclude target and static fields used for validation)
feature_cols = [
    't2m_mean_dec24_26',
    't2m_min_dec24_26',
    'precip_snow_accumulated',
    'pdd_pre_christmas',
    't2m_elevation_corrected',
    'elevation',
    'cvh',
    'cvl',
    'total_vegetation'
]

def prepare_ml_data(features_ds, feature_cols):
    """
    Reshape xarray Dataset to sklearn-compatible format.
    Returns: X (n_samples x n_features), y (n_samples,)
    """
    n_years = len(features_ds['year'])
    n_lat = len(features_ds['latitude'])
    n_lon = len(features_ds['longitude'])
    
    stacked = []
    for feat in feature_cols:
        print(f"Reshaping feature: {feat}...")
        da = features_ds[feat]
        if "year" not in da.dims:
            da = da.expand_dims(year=features_ds["year"])
        arr = da.values.reshape(n_years, -1)  # (year, spatial)
        stacked.append(arr)
    
    X = np.vstack(stacked).T  # (n_years*n_lat*n_lon, n_features)
    X = X.reshape(-1, len(feature_cols))
    
    y = features_ds['snow_presence'].values.reshape(-1)
    
    valid_idx = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X = X[valid_idx]
    y = y[valid_idx]
    
    print(f"Prepared {X.shape[0]:,} samples with {X.shape[1]} features")
    print(f"Class balance: {y.mean()*100:.1f}% snow (n={y.sum():.0f})")
    return X, y

print("Preparing training data (1980-2014)...")
X_train, y_train = prepare_ml_data(train_years, feature_cols)
print(f"Training samples: {len(X_train)}, Snow: {y_train.sum():.0f} ({100*y_train.mean():.1f}%)")

print("Preparing validation data (2015-2024)...")
X_val, y_val = prepare_ml_data(val_years, feature_cols)
print(f"Validation samples: {len(X_val)}, Snow: {y_val.sum():.0f} ({100*y_val.mean():.1f}%)")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
feature_cols

In [ ]:
# Model 1: Logistic Regression
print("\n" + "="*60)
print("MODEL 1: LOGISTIC REGRESSION")
print("="*60)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_train = lr_model.predict_proba(X_train_scaled)[:, 1]
y_pred_lr_val = lr_model.predict_proba(X_val_scaled)[:, 1]

# Metrics
auc_lr_train = roc_auc_score(y_train, y_pred_lr_train)
auc_lr_val = roc_auc_score(y_val, y_pred_lr_val)
brier_lr_val = brier_score_loss(y_val, y_pred_lr_val)

print(f"\nTraining ROC-AUC:   {auc_lr_train:.4f}")
print(f"Validation ROC-AUC: {auc_lr_val:.4f}")
print(f"Validation Brier:   {brier_lr_val:.4f}")
print(f"Feature importance (top 5):")
feat_importance_lr = sorted(zip(feature_cols, np.abs(lr_model.coef_[0])), key=lambda x: x[1], reverse=True)
for feat, imp in feat_importance_lr[:5]:
    print(f"  {feat}: {imp:.4f}")

In [ ]:
# Model 2: Random Forest Classifier
print("\n" + "="*60)
print("MODEL 2: RANDOM FOREST")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_rf_train = rf_model.predict_proba(X_train_scaled)[:, 1]
y_pred_rf_val = rf_model.predict_proba(X_val_scaled)[:, 1]

# Metrics
auc_rf_train = roc_auc_score(y_train, y_pred_rf_train)
auc_rf_val = roc_auc_score(y_val, y_pred_rf_val)
brier_rf_val = brier_score_loss(y_val, y_pred_rf_val)

print(f"\nTraining ROC-AUC:   {auc_rf_train:.4f}")
print(f"Validation ROC-AUC: {auc_rf_val:.4f}")
print(f"Validation Brier:   {brier_rf_val:.4f}")
print(f"Feature importance (top 5):")
feat_importance_rf = sorted(zip(feature_cols, rf_model.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in feat_importance_rf[:5]:
    print(f"  {feat}: {imp:.4f}")

In [ ]:
# Model 3: Physical Degree-Day Model
print("\n" + "="*60)
print("MODEL 3: PHYSICAL DEGREE-DAY MODEL")
print("="*60)

def physical_snow_model(t2m_mean, precip_snow, pdd, elevation, cvh):
    """
    Physical snow model based on temperature, precipitation, and elevation.
    Returns probability of snow presence (0-1).
    
    Logic:
    - Accumulation: Snow from precipitation when T < 2°C
    - Melt: Positive degree-days (3 mm/°C/day)
    - Persistence: Snow persists if melt + recent precip >= threshold
    - Vegetation: Dense vegetation reduces observable snow
    """
    
    # Snow water equivalent (SWE) from precipitation
    swe = precip_snow / 100  # Convert mm to cm
    
    # Melt amount (degree-day factor: 3 mm/°C/day)
    melt = np.maximum(pdd * 0.3, 0) / 100  # Convert to cm
    
    # Net snow (accumulation - melt)
    net_snow = np.maximum(swe - melt, 0)
    
    # Elevation adjustment (higher elevations more likely to have snow)
    elev_factor = 1 + (elevation / 1000) * 0.2  # +20% per 1000m
    
    # Vegetation effect (dense vegetation reduces detection)
    veg_factor = 1 - 0.3 * cvh
    
    # Probability based on snow depth threshold
    # P(snow) = logistic(net_snow * factors)
    net_snow_effective = net_snow * elev_factor * veg_factor
    
    # Logistic transform (t = net_snow, k = steepness = 5, x0 = threshold = 1cm)
    prob_snow = 1 / (1 + np.exp(-5 * (net_snow_effective - 1)))
    
    return np.clip(prob_snow, 0, 1)

# Reshape validation data back to spatial format for prediction
n_years_val = len(val_years['year'])
n_lat = len(val_years['latitude'])
n_lon = len(val_years['longitude'])

# Extract features in original shape
t2m_mean_val = val_years['t2m_mean_dec24_26'].values.reshape(-1)
precip_snow_val = val_years['precip_snow_accumulated'].values.reshape(-1)
pdd_val = val_years['pdd_pre_christmas'].values.reshape(-1)
elev_val = val_years['elevation'].values.reshape(-1)
cvh_val = val_years['cvh'].values.reshape(-1)

# Apply physical model
y_pred_physical_val = physical_snow_model(
    t2m_mean_val, 
    precip_snow_val, 
    pdd_val, 
    elev_val, 
    cvh_val
)

# Similarly for training
n_years_train = len(train_years['year'])
t2m_mean_train = train_years['t2m_mean_dec24_26'].values.reshape(-1)
precip_snow_train = train_years['precip_snow_accumulated'].values.reshape(-1)
pdd_train = train_years['pdd_pre_christmas'].values.reshape(-1)
elev_train = train_years['elevation'].values.reshape(-1)
cvh_train = train_years['cvh'].values.reshape(-1)

y_pred_physical_train = physical_snow_model(
    t2m_mean_train,
    precip_snow_train,
    pdd_train,
    elev_train,
    cvh_train
)

# Filter to valid indices (match ML validation data)
valid_idx_train = ~(np.isnan(X_train).any(axis=1) | np.isnan(y_train))
valid_idx_val = ~(np.isnan(X_val).any(axis=1) | np.isnan(y_val))

y_pred_physical_train_filtered = y_pred_physical_train[valid_idx_train]
y_pred_physical_val_filtered = y_pred_physical_val[valid_idx_val]

# Metrics
auc_phys_train = roc_auc_score(y_train, y_pred_physical_train_filtered)
auc_phys_val = roc_auc_score(y_val, y_pred_physical_val_filtered)
brier_phys_val = brier_score_loss(y_val, y_pred_physical_val_filtered)

print(f"\nTraining ROC-AUC:   {auc_phys_train:.4f}")
print(f"Validation ROC-AUC: {auc_phys_val:.4f}")
print(f"Validation Brier:   {brier_phys_val:.4f}")

In [ ]:
# Model Comparison Summary
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

comparison_df = {
    'Model': ['Logistic Regression', 'Random Forest', 'Physical Degree-Day'],
    'Train AUC': [auc_lr_train, auc_rf_train, auc_phys_train],
    'Val AUC': [auc_lr_val, auc_rf_val, auc_phys_val],
    'Val Brier': [brier_lr_val, brier_rf_val, brier_phys_val],
}

import pandas as pd
df_comp = pd.DataFrame(comparison_df)
print("\n", df_comp.to_string(index=False))

# Select best model based on validation AUC
best_model_idx = np.argmax(df_comp['Val AUC'])
best_model_name = df_comp['Model'].iloc[best_model_idx]
print(f"\n✓ Best model: {best_model_name} (Val AUC: {df_comp['Val AUC'].iloc[best_model_idx]:.4f})")

# Store best model and predictions for later use
if best_model_idx == 0:
    best_model = lr_model
    y_pred_best_train = y_pred_lr_train
    y_pred_best_val = y_pred_lr_val
    best_model_type = 'logistic'
elif best_model_idx == 1:
    best_model = rf_model
    y_pred_best_train = y_pred_rf_train
    y_pred_best_val = y_pred_rf_val
    best_model_type = 'random_forest'
else:
    # Physical model doesn't have a sklearn object, store as function
    best_model = None
    y_pred_best_train = y_pred_physical_train_filtered
    y_pred_best_val = y_pred_physical_val_filtered
    best_model_type = 'physical'

print(f"Best model type: {best_model_type}")

In [ ]:
# ROC Curve Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_val, y_pred_lr_val)
fpr_rf, tpr_rf, _ = roc_curve(y_val, y_pred_rf_val)
fpr_phys, tpr_phys, _ = roc_curve(y_val, y_pred_physical_val_filtered)

axes[0].plot(fpr_lr, tpr_lr, label=f'Logistic (AUC={auc_lr_val:.3f})', linewidth=2)
axes[0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf_val:.3f})', linewidth=2)
axes[0].plot(fpr_phys, tpr_phys, label=f'Physical (AUC={auc_phys_val:.3f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves (Validation 2015-2024)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Calibration: Predicted probability vs observed frequency
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)

for pred, label, color in [
    (y_pred_lr_val, 'Logistic', 'C0'),
    (y_pred_rf_val, 'Random Forest', 'C1'),
    (y_pred_physical_val_filtered, 'Physical', 'C2')
]:
    bin_centers = []
    obs_freq = []
    for i in range(n_bins):
        mask = (pred >= bin_edges[i]) & (pred < bin_edges[i+1])
        if mask.sum() > 0:
            bin_centers.append((bin_edges[i] + bin_edges[i+1]) / 2)
            obs_freq.append(y_val[mask].mean())
    
    axes[1].plot(bin_centers, obs_freq, 'o-', label=label, color=color, linewidth=2)

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Observed Frequency')
axes[1].set_title('Calibration Curve (Validation 2015-2024)')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('snow_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved model comparison plot to 'snow_model_comparison.png'")

## 6. CMIP6 Data Loading and Delta-Method Bias Correction

Load CMIP6 daily temperature and precipitation for historical (1980-2014) and SSP2-4.5 (2031-2100) periods. Apply delta-method bias correction by computing climatological anomalies.

In [ ]:
def load_cmip6_files(cmip6_dir, pattern, scenario='ssp245'):
    """
    Load CMIP6 NetCDF files matching pattern from a directory.
    
    Args:
        cmip6_dir: Directory containing CMIP6 files
        pattern: Filename pattern to match
        scenario: 'historical' or 'ssp245'
    
    Returns:
        xr.Dataset with concatenated files
    """
    from pathlib import Path
    import glob
    
    cmip6_path = Path(cmip6_dir)
    matching_files = list(cmip6_path.glob(pattern))
    
    if not matching_files:
        print(f"Warning: No files matching {pattern} in {cmip6_dir}")
        return None
    
    print(f"Found {len(matching_files)} files matching {pattern}")
    
    datasets = []
    for fpath in sorted(matching_files)[:5]:  # Load first 5 for demo (adjust as needed)
        print(f"  Loading {fpath.name}...", end=" ")
        try:
            ds = xr.open_dataset(fpath)
            datasets.append(ds)
            print("✓")
        except Exception as e:
            print(f"Error: {e}")
    
    if datasets:
        return xr.concat(datasets, dim='time')
    return None

# Paths
cmip6_base_dir = r"C:\Users\AlexanderKoch\TransparentEarth\lx3\data\CMIP6"

print("Loading CMIP6 historical data (1980-2014)...")
# Adjust pattern based on actual CMIP6 file structure
cmip6_hist = load_cmip6_files(cmip6_base_dir, '*historical*.nc', scenario='historical')

print("\nLoading CMIP6 SSP2-4.5 data (2031-2100)...")
cmip6_future = load_cmip6_files(cmip6_base_dir, '*ssp245*.nc', scenario='ssp245')

if cmip6_hist is not None:
    print(f"\nHistorical data shape: {dict(cmip6_hist.dims)}")
    print(f"Historical variables: {list(cmip6_hist.data_vars)}")

if cmip6_future is not None:
    print(f"\nFuture data shape: {dict(cmip6_future.dims)}")
    print(f"Future variables: {list(cmip6_future.data_vars)}")

In [ ]:
def apply_delta_method_bias_correction(
    era5_data,
    cmip6_hist,
    cmip6_future,
    variable='tas',  # or 'pr' for precipitation
    era5_var_name='t2m',
    december_only=True,
):
    """
    Apply delta-method bias correction to CMIP6 future data.
    
    Method:
    1. Compute Dec climatology from ERA5-Land (1980-2014)
    2. Compute Dec climatology from CMIP6 historical (1980-2014)
    3. Compute anomaly: CMIP6_future - CMIP6_hist_clim
    4. Correct future = ERA5_clim + anomaly
    
    Args:
        era5_data: ERA5-Land xarray Dataset (daily)
        cmip6_hist: CMIP6 historical xarray Dataset (daily)
        cmip6_future: CMIP6 future xarray Dataset (daily)
        variable: Variable name in CMIP6 ('tas', 'pr')
        era5_var_name: Variable name in ERA5 ('t2m', 'tp')
        december_only: Use only December data
    
    Returns:
        Bias-corrected CMIP6 future data
    """
    
    print(f"Applying delta-method bias correction for {variable}...")
    
    # --- ERA5 climatology (Dec only, 1980-2014) ---
    era5_dec = era5_data[era5_var_name].sel(time=slice('1980-01-01', '2014-12-31'))
    era5_dec = era5_dec.where(era5_dec['time'].dt.month == 12, drop=True)
    
    # Compute daily climatology (mean of each day across all years)
    era5_dec_day = era5_dec.groupby('time.day').mean('time')
    era5_clim = era5_dec_day.mean('day')  # Simpler: just use mean across Dec
    
    print(f"  ERA5 Dec climatology: mean={era5_clim.values.mean():.2f}, std={era5_clim.values.std():.2f}")
    
    # --- CMIP6 historical climatology (Dec only, 1980-2014) ---
    if cmip6_hist is not None:
        cmip6_hist_dec = cmip6_hist[variable].sel(time=slice('1980-01-01', '2014-12-31'))
        cmip6_hist_dec = cmip6_hist_dec.where(cmip6_hist_dec['time'].dt.month == 12, drop=True)
        cmip6_hist_clim = cmip6_hist_dec.mean('time')
        
        print(f"  CMIP6 Hist Dec climatology: mean={cmip6_hist_clim.values.mean():.2f}")
    else:
        print("  Warning: CMIP6 historical not available, using CMIP6 future for anomalies")
        cmip6_hist_clim = cmip6_future[variable].sel(time=slice('2031-01-01', '2060-12-31')).mean('time')
    
    # --- CMIP6 future anomaly (relative to historical climatology) ---
    cmip6_future_dec = cmip6_future[variable].where(cmip6_future[variable]['time'].dt.month == 12, drop=True)
    cmip6_future_clim = cmip6_future_dec.mean('time')  # Overall future climatology
    
    # Anomaly relative to historical
    anomaly = cmip6_future_clim - cmip6_hist_clim
    
    print(f"  Anomaly: mean={anomaly.values.mean():.2f}")
    
    # --- Bias-corrected future data ---
    cmip6_corrected = cmip6_future[variable] + anomaly
    
    print(f"  ✓ Bias correction applied")
    
    return cmip6_corrected, anomaly, era5_clim

# Apply delta method to temperature and precipitation if available
if cmip6_hist is not None and cmip6_future is not None:
    print("=" * 60)
    print("DELTA-METHOD BIAS CORRECTION")
    print("=" * 60)
    
    # Identify variable names in CMIP6 (might be 'tas', 'Tas', 'temperature', etc.)
    cmip6_vars = list(cmip6_future.data_vars)
    print(f"\nAvailable CMIP6 variables: {cmip6_vars}")
    
    # Temperature correction
    if 'tas' in cmip6_vars or 'Tas' in cmip6_vars:
        tas_var = 'tas' if 'tas' in cmip6_vars else 'Tas'
        cmip6_t_corrected, t_anomaly, era5_t_clim = apply_delta_method_bias_correction(
            ds_era5, cmip6_hist, cmip6_future,
            variable=tas_var,
            era5_var_name='t2m',
            december_only=True
        )
    
    # Precipitation correction
    if 'pr' in cmip6_vars or 'Pr' in cmip6_vars:
        pr_var = 'pr' if 'pr' in cmip6_vars else 'Pr'
        cmip6_pr_corrected, pr_anomaly, era5_pr_clim = apply_delta_method_bias_correction(
            ds_era5, cmip6_hist, cmip6_future,
            variable=pr_var,
            era5_var_name='tp',
            december_only=True
        )
else:
    print("Warning: CMIP6 data not fully available, skipping bias correction for now")

## 7. Future White Christmas Projections

Apply best-performing snow model to bias-corrected CMIP6 data. Generate probability maps for future periods (2040-2049, 2060-2069, 2090-2099) and compute change relative to historical baseline (1980-2024).

In [ ]:
def project_white_christmas_probability(
    t2m_daily,
    tp_daily,
    elevation,
    cvh,
    cvl,
    best_model,
    best_model_type,
    scaler,
    feature_cols,
):
    """
    Compute white Christmas probability for CMIP6 future data.
    
    Args:
        t2m_daily: Daily temperature (K)
        tp_daily: Daily precipitation (m)
        elevation: Elevation (m)
        cvh: High vegetation cover (0-1)
        cvl: Low vegetation cover (0-1)
        best_model: Fitted sklearn model (or None for physical)
        best_model_type: 'logistic', 'random_forest', or 'physical'
        scaler: StandardScaler fitted on training data
        feature_cols: List of feature column names
    
    Returns:
        xr.DataArray of white Christmas probability (0-1)
    """
    
    # Extract December Dec 24-26 data
    t2m_c = t2m_daily - 273.15
    tp_mm = tp_daily * 1000
    
    # Filter to Dec 24-26
    t2m_dec24_26 = t2m_c.where(
        (t2m_c['time'].dt.month == 12) & 
        (t2m_c['time'].dt.day.isin([24, 25, 26])),
        drop=True
    )
    
    if len(t2m_dec24_26['time']) == 0:
        print("No Dec 24-26 data found")
        return None
    
    # Compute features
    t2m_mean = t2m_dec24_26.mean('time', skipna=True)
    t2m_min = t2m_dec24_26.min('time', skipna=True)
    
    # Pre-Christmas (Dec 10-23) features
    t2m_pre = t2m_c.where(
        (t2m_c['time'].dt.month == 12) &
        (t2m_c['time'].dt.day >= 10) &
        (t2m_c['time'].dt.day <= 23),
        drop=True
    )
    tp_pre = tp_mm.where(
        (tp_mm['time'].dt.month == 12) &
        (tp_mm['time'].dt.day >= 10) &
        (tp_mm['time'].dt.day <= 23),
        drop=True
    )
    
    is_snow = t2m_pre < 2.0
    precip_snow = (tp_pre * is_snow).sum('time', skipna=True)
    pdd = (t2m_pre.clip(min=0)).sum('time', skipna=True)
    
    # Elevation-corrected temperature
    t2m_corrected = t2m_mean + (elevation * 0.0065)
    
    # Total vegetation
    total_veg = cvh + cvl
    
    if best_model_type == 'physical':
        # Use physical model
        prob = physical_snow_model(
            t2m_mean.values,
            precip_snow.values,
            pdd.values,
            elevation.values,
            cvh.values
        )
        prob_da = xr.DataArray(
            prob.reshape(t2m_mean.shape),
            coords=t2m_mean.coords,
            dims=t2m_mean.dims,
            name='snow_probability'
        )
    else:
        # Use ML model (logistic or RF)
        # Create feature array
        n_lat = len(t2m_mean['latitude'])
        n_lon = len(t2m_mean['longitude'])
        
        X_feat = np.column_stack([
            t2m_mean.values.flatten(),
            t2m_min.values.flatten(),
            precip_snow.values.flatten(),
            pdd.values.flatten(),
            t2m_corrected.values.flatten(),
            elevation.values.flatten(),
            cvh.values.flatten(),
            cvl.values.flatten(),
            total_veg.values.flatten(),
        ])
        
        # Scale and predict
        X_scaled = scaler.transform(X_feat)
        prob = best_model.predict_proba(X_scaled)[:, 1]
        
        prob_da = xr.DataArray(
            prob.reshape(t2m_mean.shape),
            coords=t2m_mean.coords,
            dims=t2m_mean.dims,
            name='snow_probability'
        )
    
    return prob_da


# Define future periods for projection
future_periods = [
    (2040, 2049, '2040_2049'),
    (2060, 2069, '2060_2069'),
    (2090, 2099, '2090_2099'),
]

if cmip6_future is not None:
    print("\n" + "="*60)
    print("FUTURE WHITE CHRISTMAS PROJECTIONS (CMIP6 SSP2-4.5)")
    print("="*60)
    
    future_projections = {}
    
    for start_year, end_year, label in future_periods:
        print(f"\nProjecting {label}...")
        
        # Select time period
        try:
            t2m_period = cmip6_t_corrected.sel(time=slice(f'{start_year}-01-01', f'{end_year}-12-31'))
            tp_period = cmip6_pr_corrected.sel(time=slice(f'{start_year}-01-01', f'{end_year}-12-31'))
        except:
            print(f"  Warning: Data not available for {label}, skipping")
            continue
        
        # Project probability
        proj_prob = project_white_christmas_probability(
            t2m_period,
            tp_period,
            elevation_aligned,
            cvh_aligned,
            cvl_aligned,
            best_model,
            best_model_type,
            scaler,
            feature_cols
        )
        
        if proj_prob is not None:
            # Average over period
            future_projections[label] = proj_prob.mean('time', skipna=True) * 100  # Convert to percent
            
            print(f"  Mean probability: {future_projections[label].values.mean():.1f}%")
            
            # Compute change relative to 1980-2024 baseline
            baseline_prob = results['1980_2024']['p_white_pct']
            delta = future_projections[label] - baseline_prob
            
            # Export change map
            delta_tif = to_geotiff_ready(delta)
            delta_tif.name = f'delta_prob_pp_{label}'
            delta_tif.rio.to_raster(
                f'white_christmas_delta_prob_pp_{label}.tif',
                compress="DEFLATE",
                dtype="float32",
                nodata=np.nan,
            )
            
            # Export probability map
            prob_tif = to_geotiff_ready(future_projections[label])
            prob_tif.name = f'probability_{label}'
            prob_tif.rio.to_raster(
                f'white_christmas_probability_pct_{label}.tif',
                compress="DEFLATE",
                dtype="float32",
                nodata=np.nan,
            )
            
            print(f"  Change from baseline: {delta.values.mean():.1f} pp")
            print(f"  ✓ Exported GeoTIFFs")
    
    print("\n✓ Future projections complete")
else:
    print("\nWarning: CMIP6 future data not available, skipping projections")

## 8. Summary and Output

Export final summary statistics and visualization of historical trends and future projections.

In [ ]:
# Create summary table of historical and future probabilities
print("\n" + "="*60)
print("SUMMARY: WHITE CHRISTMAS PROBABILITY (%) BY PERIOD")
print("="*60)

summary_data = []

# Historical periods
for label, result_ds in results.items():
    prob_mean = result_ds['p_white_pct'].mean().values
    prob_std = result_ds['p_white_pct'].std().values
    summary_data.append({
        'Period': label,
        'Type': 'Historical (ERA5)',
        'Mean Probability (%)': f"{prob_mean:.1f}",
        'Std Dev (%)': f"{prob_std:.1f}",
    })

# Future periods
if 'future_projections' in locals():
    baseline_prob_mean = results['1980_2024']['p_white_pct'].mean().values
    
    for label, prob_da in future_projections.items():
        proj_mean = prob_da.mean().values
        delta_from_baseline = proj_mean - baseline_prob_mean
        summary_data.append({
            'Period': label,
            'Type': 'Future (CMIP6 SSP2-4.5)',
            'Mean Probability (%)': f"{proj_mean:.1f}",
            'Change from 1980-2024 (pp)': f"{delta_from_baseline:.1f}",
        })

df_summary = pd.DataFrame(summary_data)
print("\n", df_summary.to_string(index=False))

# Export summary
df_summary.to_csv('white_christmas_summary.csv', index=False)
print("\n✓ Summary exported to 'white_christmas_summary.csv'")

# List all output files
print("\n" + "="*60)
print("OUTPUT FILES GENERATED")
print("="*60)

output_files = [
    "Historical probability maps (GeoTIFF): white_christmas_probability_pct_*.tif",
    "Historical trend maps (GeoTIFF): white_christmas_trend_pp_per_decade_*.tif",
    "Historical valid years (GeoTIFF): white_christmas_valid_years_*.tif",
    "Future probability maps (GeoTIFF): white_christmas_probability_pct_2040_2049.tif, etc.",
    "Future change maps (GeoTIFF): white_christmas_delta_prob_pp_2040_2049.tif, etc.",
    "Model comparison plot: snow_model_comparison.png",
    "Summary table: white_christmas_summary.csv",
]

print("\nGenerated outputs:")
for i, fname in enumerate(output_files, 1):
    print(f"  {i}. {fname}")

print("\n" + "="*60)
print("✓ ANALYSIS COMPLETE")
print("="*60)